In [0]:
from pyspark.sql import functions as F
from config import TABLES, CHECKPOINTS
from metrics_utils import log_pipeline_metric

In [0]:
df_eventhub = (
    spark.readStream.table(TABLES["bronze_eventhub"])
    .withColumn("source", F.lit("eventhub"))
)

df_files = (
    spark.readStream.table(TABLES["bronze_files"])
    .withColumn("source", F.lit("files"))
)

In [0]:
df_bronze = df_eventhub.unionByName(df_files, allowMissingColumns=True)

In [0]:
df_bronze = df_bronze.select(
    F.col("InvoiceNo").alias("invoice_no"),
    F.col("StockCode").alias("stock_code"),
    F.col("Description").alias("description"),
    F.col("Quantity").alias("quantity"),
    F.col("InvoiceDate").alias("invoice_date"),
    F.col("UnitPrice").alias("unit_price"),
    F.col("CustomerID").alias("customer_id"),
    F.col("Country").alias("country"),
    F.col("_ingest_ts"),
    F.col("_source_file"),
    F.col("source")
).withColumn(
    "pipeline_stage", F.lit("silver")
)

In [0]:
df_clean = (
    df_bronze
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .withColumn("invoice_ts", F.to_timestamp("invoice_date", "M/d/yyyy H:mm"))
    .withColumn("invoice_dt", F.to_date("invoice_ts"))
    .withColumn("year", F.year("invoice_dt"))
    .withColumn("month", F.month("invoice_dt"))
    .withColumn("day", F.dayofmonth("invoice_dt"))
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
)

In [0]:
silver_cols = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "invoice_ts",
    "invoice_dt",
    "_ingest_ts",
    "_source_file",
    "revenue",
    "year",
    "month",
    "day",
    "pipeline_stage"
]

df_silver = df_clean.select(silver_cols)

In [0]:
df_silver = (
    df_silver
    .withWatermark("invoice_ts", "2 hours")
    .dropDuplicates(["invoice_no","stock_code","customer_id","invoice_date"])
)

In [0]:
from pyspark.sql import functions as F

def log_pipeline_metric(spark, pipeline_name, df):

    metrics_df = spark.createDataFrame(
        [(pipeline_name, df.count())],
        ["pipeline", "row_count"]
    ).withColumn("processed_at", F.current_timestamp())

    metrics_df.write.mode("append").saveAsTable(
        "dbw_retails.monitoring.pipeline_metrics"
    )

In [0]:
def process_batch(batch_df, batch_id):

    from config import TABLES
    from metrics_utils import log_pipeline_metric

    (
        batch_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLES["silver"])
    )

    log_pipeline_metric(spark, "silver_unified", batch_df)

In [0]:
query = (
    df_silver.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINTS["silver"])
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()